In [ ]:
# Tensors in Practice
# Generated from the canonical HTML manuscript. Run this cell first.
# Source: https://github.com/Shakeri-Lab/dl-book/blob/c058d1f401fd0ead3ae59a2a8730f95489a2d9aa/chapters/appendices/a2-tensors.qmd

from importlib.metadata import PackageNotFoundError, version as package_version
import hashlib as _bootstrap_hashlib
import os as _bootstrap_os
from pathlib import Path as _BootstrapPath
import subprocess as _bootstrap_subprocess
import sys as _bootstrap_sys
import urllib.request as _bootstrap_urlrequest

_BOOK_REVISION = 'c058d1f401fd0ead3ae59a2a8730f95489a2d9aa'
_PINNED_REQUIREMENTS = [
    "torch==2.12.1",
    "torchvision==0.27.1",
    "numpy==2.5.1",
    "matplotlib==3.11.1"
]
_BOOK_ASSETS = []

def _installed_requirement(requirement: str) -> bool:
    name, expected = requirement.split('==', 1)
    try:
        return package_version(name) == expected
    except PackageNotFoundError:
        return False

_missing_requirements = [
    item for item in _PINNED_REQUIREMENTS if not _installed_requirement(item)
]
if _missing_requirements:
    _bootstrap_install = _bootstrap_subprocess.run(
        [_bootstrap_sys.executable, '-m', 'pip', 'install', '--quiet',
         *_missing_requirements],
        check=False, capture_output=True, text=True,
    )
    if _bootstrap_install.returncode != 0:
        raise RuntimeError(_bootstrap_install.stdout + _bootstrap_install.stderr)

_bootstrap_base = _BootstrapPath(
    _bootstrap_os.environ.get(
        'DLBOOK_NOTEBOOK_ROOT',
        '/content' if _BootstrapPath('/content').is_dir()
        else str(_BootstrapPath.home() / '.cache'),
    )
)
_BOOK_ROOT = _bootstrap_base / f'dl-book-{_BOOK_REVISION[:12]}'
_RAW_ROOT = 'https://raw.githubusercontent.com/Shakeri-Lab/dl-book/' + _BOOK_REVISION + '/'
for _record in _BOOK_ASSETS:
    _destination = _BOOK_ROOT / _record['path']
    _destination.parent.mkdir(parents=True, exist_ok=True)
    _valid = (
        _destination.is_file()
        and _bootstrap_hashlib.sha256(_destination.read_bytes()).hexdigest()
        == _record['sha256']
    )
    if not _valid:
        _temporary = _destination.with_suffix(_destination.suffix + '.part')
        _bootstrap_urlrequest.urlretrieve(_RAW_ROOT + _record['path'], _temporary)
        _digest = _bootstrap_hashlib.sha256(_temporary.read_bytes()).hexdigest()
        if _digest != _record['sha256']:
            _temporary.unlink(missing_ok=True)
            raise RuntimeError(f"Checksum mismatch for {_record['path']}")
        _temporary.replace(_destination)

(_BOOK_ROOT / 'chapters/appendices').mkdir(parents=True, exist_ok=True)
_bootstrap_sys.path.insert(0, str(_BOOK_ROOT / 'code'))
_bootstrap_os.chdir(_BOOK_ROOT / 'chapters/appendices')

# Hidden manuscript support required by later learner-visible cells.
# Plot-only harnesses are not exported.
import torch
from torch import nn

assert _BOOK_ROOT.is_dir()

**Plan**

1. Prepare the inputs and fixed settings for the example.
2. Verify the nn.Linear weight and bias shape contract.
3. Report or visualize the measured result.

In [ ]:
import torch
from torch import nn

# [1]
_ = torch.manual_seed(6050)

B, D_IN, D_OUT = 3, 4, 2
X = torch.arange(B * D_IN, dtype=torch.float64).reshape(B, D_IN) / 10
W = torch.tensor(
    [[1.0, -1.0, 0.5, 2.0], [-0.5, 1.5, 1.0, 0.0]],
    dtype=torch.float64,
)
b = torch.tensor([0.25, -0.75], dtype=torch.float64)

manual = X @ W.T + b
layer = nn.Linear(D_IN, D_OUT, dtype=torch.float64)
with torch.no_grad():
    layer.weight.copy_(W)
    layer.bias.copy_(b)
output = layer(X)

# [2]
assert output.shape == (B, D_OUT)
assert torch.allclose(output, manual)
gap = (output - manual).detach().abs().max().item()
# [3]
print(f"X {tuple(X.shape)}, W {tuple(W.shape)}, b {tuple(b.shape)}")
print(f"output {tuple(output.shape)}, max gap {gap:.1f}")

**Plan**

1. Prepare the inputs and fixed settings for the example.
2. Expose a silent channel-versus-width broadcasting error.
3. Report or visualize the measured result.

In [ ]:
# [1]
images = torch.tensor(
    [[[[1.0, 1.0, 1.0], [1.0, 1.0, 1.0]],
      [[2.0, 2.0, 2.0], [2.0, 2.0, 2.0]],
      [[3.0, 3.0, 3.0], [3.0, 3.0, 3.0]]]]
)  # (B=1, C=3, H=2, W=3)

channel_mean = images.mean(dim=(0, 2, 3), keepdim=True)
centered = images - channel_mean
wrong = images - channel_mean.reshape(3)

correct_check = centered.mean(dim=(0, 2, 3))
wrong_check = wrong.mean(dim=(0, 2, 3))
# [2]
assert torch.equal(correct_check, torch.zeros(3))
assert not torch.equal(wrong_check, torch.zeros(3))

# [3]
print(f"images {tuple(images.shape)}, mean {tuple(channel_mean.shape)}")
print("correct channel means:", correct_check.tolist())
print("wrong channel means:  ", wrong_check.tolist())

**Plan**

1. Prepare the inputs and fixed settings for the example.
2. Contrast an expanded view with repeated storage.
3. Report or visualize the measured result.

In [ ]:
# [1]
base = torch.tensor([[10.0], [20.0]])       # (2, 1)
expanded = base.expand(2, 3)                # view: stride 0 on the new width
repeated = base.repeat(1, 3)                # materialized tiled values

base[0, 0] = 99.0
# [2]
assert expanded[0].tolist() == [99.0, 99.0, 99.0]
assert repeated[0].tolist() == [10.0, 10.0, 10.0]

# [3]
print("expanded stride:", expanded.stride())
print("repeated stride:", repeated.stride())
print("after editing base:", expanded[0].tolist(), repeated[0].tolist())

**Plan**

1. Prepare the inputs and fixed settings for the example.
2. Follow shape and stride through a permutation.
3. Report or visualize the measured result.

In [ ]:
# [1]
original = torch.arange(24).reshape(2, 3, 4)
permuted = original.permute(0, 2, 1)

try:
    permuted.view(2, -1)
except RuntimeError:
    view_result = "view rejected the incompatible strides"
else:
    raise AssertionError("view unexpectedly succeeded")

reshaped = permuted.reshape(2, -1)
materialized = permuted.contiguous().view(2, -1)
# [2]
assert torch.equal(reshaped, materialized)
reshape_copied = (
    reshaped.untyped_storage().data_ptr()
    != permuted.untyped_storage().data_ptr()
)

# [3]
print("original:", tuple(original.shape), original.stride())
print("permuted:", tuple(permuted.shape), permuted.stride(),
      permuted.is_contiguous())
print(view_result)
print("reshape copied in this case:", reshape_copied)

**Plan**

1. Prepare the inputs and fixed settings for the example.
2. Verify batched matmul against an indexed contraction.
3. Report or visualize the measured result.

In [ ]:
# [1]
generator = torch.Generator().manual_seed(6050)
B, N_HEADS, T, S, D_HEAD = 2, 3, 4, 5, 6
queries = torch.randn(
    B, N_HEADS, T, D_HEAD, generator=generator, dtype=torch.float64
)
keys = torch.randn(
    B, N_HEADS, S, D_HEAD, generator=generator, dtype=torch.float64
)

by_matmul = queries @ keys.transpose(-2, -1)
by_indices = torch.einsum("bhtd,bhsd->bhts", queries, keys)

# [2]
assert by_matmul.shape == (B, N_HEADS, T, S)
assert torch.allclose(by_matmul, by_indices, rtol=0, atol=1e-12)
gap = (by_matmul - by_indices).abs().max().item()
# [3]
print(
    f"Q {tuple(queries.shape)}, K {tuple(keys.shape)}, "
    f"scores {tuple(by_matmul.shape)}"
)
print(f"matmul/einsum max gap: {gap:.1f}")

**Plan**

1. Prepare the inputs and fixed settings for the example.
2. Contrast Boolean selection with shape-preserving masking.
3. Report or visualize the measured result.

In [ ]:
# [1]
tokens = torch.arange(2 * 4 * 3).reshape(2, 4, 3)
visible = torch.tensor(
    [[True, True, False, False], [True, False, True, False]]
)

selected = tokens[visible]
masked = tokens.masked_fill(~visible[..., None], -1)

# [2]
assert selected.shape == (4, 3)
assert masked.shape == tokens.shape
assert torch.equal(masked[visible], selected)
# [3]
print(f"tokens {tuple(tokens.shape)}, mask {tuple(visible.shape)}")
print(f"selection {tuple(selected.shape)}, masked tensor {tuple(masked.shape)}")

**Plan**

1. Prepare the inputs and fixed settings for the example.
2. Construct values, masks, and indices from one reference.
3. Report or visualize the measured result.

In [ ]:
# [1]
reference = torch.linspace(-1, 1, 6, dtype=torch.float64).reshape(2, 3)
reference.requires_grad_(True)

accumulator = torch.zeros_like(reference)
thresholds = reference.new_full((1, 3), 0.25)
valid = torch.ones(
    reference.shape[0], dtype=torch.bool, device=reference.device
)
indices = torch.arange(reference.shape[0], device=reference.device)

# [2]
assert accumulator.dtype == thresholds.dtype == reference.dtype
assert accumulator.device == thresholds.device == reference.device
assert not accumulator.requires_grad
assert valid.dtype == torch.bool and indices.dtype == torch.int64

# [3]
print("reference:", reference.dtype, reference.device, reference.requires_grad)
print("accumulator:", accumulator.dtype, accumulator.device,
      accumulator.requires_grad)
print("semantic dtypes:", valid.dtype, indices.dtype)